In [1]:
import pandas as pd
import numpy as np
import imblearn

In [2]:
data = pd.read_csv('aug_test.csv')
data.head()

,id,Gender,Age,Driving_License,Region_Code,Previously_Insured,Vehicle_Age,Vehicle_Damage,Annual_Premium,Policy_Sales_Channel,Vintage
0,57782,Female,34,1,39.0,1,1-2 Year,No,38244.0,124.0,146
1,286811,Female,55,1,28.0,0,> 2 Years,Yes,37577.0,122.0,109
2,117823,Male,39,1,28.0,1,1-2 Year,No,24578.0,26.0,63
3,213992,Male,28,1,50.0,1,1-2 Year,No,40507.0,8.0,129
4,324756,Female,24,1,10.0,0,< 1 Year,Yes,36783.0,152.0,201


In [4]:
data.columns

Index(['id', 'Gender', 'Age', 'Driving_License', 'Region_Code',
       'Previously_Insured', 'Vehicle_Age', 'Vehicle_Damage', 'Annual_Premium',
       'Policy_Sales_Channel', 'Vintage'],
      dtype='object')

In [5]:
data.describe()

,id,Age,Driving_License,Region_Code,Previously_Insured,Annual_Premium,Policy_Sales_Channel,Vintage
count,78273.000000,78273.000000,78273.000000,78273.000000,78273.000000,78273.000000,78273.000000,78273.000000
mean,233993.913827,38.507570,0.997866,26.381434,0.488917,30707.042441,111.993216,154.827220
std,139265.743227,15.216589,0.046141,13.149780,0.499880,17044.185877,54.270018,83.476632
min,2.000000,20.000000,0.000000,0.000000,0.000000,2630.000000,1.000000,10.000000
25%,115579.000000,25.000000,1.000000,15.000000,0.000000,24548.000000,26.000000,83.000000
50%,229110.000000,36.000000,1.000000,28.000000,0.000000,31741.000000,150.000000,155.000000
75%,344739.000000,49.000000,1.000000,35.000000,1.000000,39476.000000,152.000000,227.000000
max,508136.000000,85.000000,1.000000,52.000000,1.000000,489663.000000,163.000000,299.000000


In [9]:
from sklearn.ensemble import HistGradientBoostingClassifier


In [7]:
X = data.iloc[:, 2:]
X.head()

,Age,Driving_License,Region_Code,Previously_Insured,Vehicle_Age,Vehicle_Damage,Annual_Premium,Policy_Sales_Channel,Vintage
0,34,1,39.0,1,1-2 Year,No,38244.0,124.0,146
1,55,1,28.0,0,> 2 Years,Yes,37577.0,122.0,109
2,39,1,28.0,1,1-2 Year,No,24578.0,26.0,63
3,28,1,50.0,1,1-2 Year,No,40507.0,8.0,129
4,24,1,10.0,0,< 1 Year,Yes,36783.0,152.0,201


In [19]:
y = data['Previously_Insured']

In [20]:
from sklearn.model_selection import  train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y)

In [15]:
#model = CatBoostClassifier(iterations=100, learning_rate=0.1, depth=6, eval_metric='Accuracy', verbose=10, task_type='GPU')
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    eval_metric='logloss',
    tree_method='gpu_hist',
    verbosity=1
)


In [22]:
from xgboost import XGBClassifier

model = XGBClassifier(
    enable_categorical=True,
    tree_method='gpu_hist',
    n_estimators=100
)

X_train[['Vehicle_Age','Vehicle_Damage']] = X_train[['Vehicle_Age','Vehicle_Damage']].astype('category')
X_test[['Vehicle_Age','Vehicle_Damage']] = X_test[['Vehicle_Age','Vehicle_Damage']].astype('category')


In [24]:
# model.fit(X_train, y_train)


In [25]:
# y_pred = model.predict(X_test)

In [26]:
from imblearn.under_sampling import RandomUnderSampler
rus = RandomUnderSampler()
X_rus, y_rus = rus.fit_resample(X, y)

In [27]:
np.shape(X_rus)

(76538, 9)

In [28]:
np.unique(y_rus, return_counts = True)

(array([0, 1]), array([38269, 38269]))

In [29]:
X_train, X_test, y_train, y_test = train_test_split(X_rus, y_rus)

In [37]:

from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=100,      # iterations
    learning_rate=0.1,
    max_depth=6,           # depth
    eval_metric='logloss',
    tree_method='hist',
    verbosity=1
)


In [38]:
X_train.columns = X_train.columns.str.replace('[\[\]<>()]', '_', regex=True)
X_test.columns  = X_test.columns.str.replace('[\[\]<>()]', '_', regex=True)

X_train = pd.get_dummies(X_train)
X_test  = pd.get_dummies(X_test)


<>:1: SyntaxWarning: invalid escape sequence '\['
<>:2: SyntaxWarning: invalid escape sequence '\['
<>:1: SyntaxWarning: invalid escape sequence '\['
<>:2: SyntaxWarning: invalid escape sequence '\['
C:\Users\amir\AppData\Local\Temp\ipykernel_3940\1893135894.py:1: SyntaxWarning: invalid escape sequence '\['
  X_train.columns = X_train.columns.str.replace('[\[\]<>()]', '_', regex=True)
C:\Users\amir\AppData\Local\Temp\ipykernel_3940\1893135894.py:2: SyntaxWarning: invalid escape sequence '\['
  X_test.columns  = X_test.columns.str.replace('[\[\]<>()]', '_', regex=True)


In [39]:
model.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None,
              num_parallel_tree=None, ...)

In [40]:
y_pred = model.predict(X_test)

In [42]:
from sklearn.metrics import classification_report
print(classification_report(y_true=y_test, y_pred=y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      9581
           1       1.00      1.00      1.00      9554

    accuracy                           1.00     19135
   macro avg       1.00      1.00      1.00     19135
weighted avg       1.00      1.00      1.00     19135



In [ ]:
from imblearn.over_sampling import RandomOverSampler
ros = R 